# Demo Colab — Detection de fraude assurance habitation (variante FULL)

**Soutenance ISFA 2025-2026 — Projet Data Mining**

Ce notebook deploie l'application Streamlit en mode **full** (CLIP + BLIP-2 + LLaVA + Mistral-7B).
Il faut un runtime GPU. Recommande : **L4** (24 Go VRAM) ou **A100**. T4 (16 Go) fonctionne en serrant.

Etapes :
1. Verifier le GPU
2. Cloner le repo GitHub
3. Installer les dependances
4. Lancer Streamlit en arriere-plan
5. Exposer via localtunnel (URL publique)

## 1. Verification du GPU

**Avant de lancer cette cellule** : menu Colab > Modifier > Parametres du notebook > Type d'execution = GPU (L4 ou A100 recommandes).
Si tu as Colab Pro, choisis L4 ou A100 dans le selecteur "GPU".

In [ ]:
!nvidia-smi

## 2. Clone du depot GitHub

Si le depot a deja ete clone (en cas de re-execution), on saute l'etape.

In [ ]:
import os
REPO_URL = "https://github.com/HpedArthur/Insurance-fraud-detection-AI.git"
REPO_DIR = "/content/Insurance-fraud-detection-AI"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print("Depot deja clone, on pull les derniers commits.")
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!ls -la 04_models/

## 3. Installation des dependances

Colab a deja PyTorch installe (pas besoin de reinstaller). On ajoute :
- `open_clip_torch` pour CLIP
- `streamlit` + `pyngrok`/`localtunnel` pour l'expo
- `bitsandbytes` pour la quantization 4-bit (LLaVA et Mistral)
- `transformers`, `accelerate` (probablement deja installes mais on s'assure)
- `imagehash` pour les metadonnees image
- `spacy` + le modele fr_core_news_md
- `imbalanced-learn`, `xgboost`, `shap` pour la pipeline complete

In [ ]:
!pip install -q streamlit==1.42.0 open_clip_torch bitsandbytes==0.43.3 \
    transformers==4.45.2 accelerate sentencepiece protobuf \
    imagehash imbalanced-learn xgboost shap \
    spacy==3.8.3
!pip install -q https://github.com/explosion/spacy-models/releases/download/fr_core_news_md-3.8.0/fr_core_news_md-3.8.0-py3-none-any.whl
print('Installation terminee')

## 4. (Optionnel) Token HuggingFace

Certains modeles (LLaVA, Mistral) requierent un compte HF gratuit + acceptation de la licence sur leur page modele.
Si c'est ton premier lancement, va sur :
- https://huggingface.co/llava-hf/llava-1.5-7b-hf (clique "Agree and access repository")
- https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 (idem)

Puis genere un token Read sur https://huggingface.co/settings/tokens et colle-le ci-dessous.

In [ ]:
from huggingface_hub import login
# Decommente la ligne ci-dessous et colle ton token HF (commence par hf_...) :
# login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
print("Authentification HF a configurer manuellement ci-dessus.")

## 5. Lancement de Streamlit en arriere-plan

On lance Streamlit avec `nohup` pour qu'il tourne en background.
Le `--server.fileWatcherType none` evite un bug Python + torch sur les imports dynamiques.

In [ ]:
import subprocess, time, os

# Tue toute instance existante
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

# Lance Streamlit en background sur le port 8501
os.chdir(REPO_DIR)
log_path = '/content/streamlit.log'
subprocess.Popen(
    ['streamlit', 'run', '05_app/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.fileWatcherType', 'none',
     '--browser.gatherUsageStats', 'false'],
    stdout=open(log_path, 'w'), stderr=subprocess.STDOUT,
)
print('Streamlit demarre. Attente du chargement (~15s)...')
time.sleep(15)
!tail -n 20 /content/streamlit.log

## 6. Expose via localtunnel

Installe `localtunnel` (Node.js) puis lance un tunnel sur le port 8501.
Localtunnel affiche un IP de tunnel (`Tunnel Password`). **Note-le**, il faudra le coller sur la page de bienvenue de localtunnel.

L'URL publique sera de la forme `https://xxxx-xxxx.loca.lt`.

In [ ]:
!npm install -g localtunnel 2>&1 | tail -3

In [ ]:
# Recuperer l'IP publique (mot de passe demande par localtunnel a la 1re visite)
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print(f'Mot de passe localtunnel (a coller sur la page d accueil) : {ip}')
print()
print('Lance la cellule suivante pour demarrer le tunnel.')

In [ ]:
# Lance localtunnel. Garde cette cellule ouverte pendant toute la duree de la demo.
# L URL publique s affiche dans la sortie ci-dessous.
!lt --port 8501

## 7. Mode d'emploi en soutenance

1. Lance les cellules 1 a 6 dans l'ordre.
2. Note l'URL `https://xxxx.loca.lt` affichee dans la cellule 6.
3. Note le mot de passe (IP) affiche dans la cellule 5.
4. Ouvre l'URL dans un nouvel onglet : tu verras la page "You are about to visit..." — colle l IP comme mot de passe, clique "Click to Submit".
5. L'app Streamlit s'affiche.
6. Dans la sidebar, selectionne **variante = full** et coche **BLIP-2 + LLaVA** et **Mistral judge**.
7. Premier upload : le chargement des 3 LMM prend **2-4 minutes** (telechargement depuis HF Hub).
8. A partir du 2e upload : ~30s par image (inclus heatmap).

## Diagnostic / problemes courants

**"CUDA out of memory"** sur T4 (16 Go) :
- Decoche "BLIP-2 + LLaVA" OU "Mistral judge", garde un seul des deux.
- Ou passe en L4/A100 dans les parametres du runtime.

**"Erreur 502" sur l'URL loca.lt** :
- Streamlit a crashe. Relance la cellule 5 et regarde `/content/streamlit.log`.

**"403 Forbidden" sur LLaVA / Mistral** :
- Tu n'as pas accepte la licence HF, ou ton token n est pas configure (cellule 4).